In [ ]:
"""
SSP / tool switching com QAOA em codificacao compacta (ranking de permutacoes).

- Simulador QAOA direto em NumPy (mixer via transformada de Walsh-Hadamard).
- Metricas de diagnostico: E[QAOA] vs E[uniforme], P(otimo) vs P(otimo uniforme),
  P(invalido), e baseline de sorteio aleatorio com o mesmo numero de avaliacoes.
- Varias sementes por tamanho; saida detalhada e resumida em CSV.

Requer Python >= 3.10 (int.bit_count).
"""
import hashlib
import itertools
import math
import os
import time

import numpy as np
import pandas as pd
from scipy.optimize import minimize


# ============================================================
# 0. Instancia e custo (SSP / tool switching, politica KTNS)
# ============================================================

def gerar_instancia(n, seed=0, fator_capacidade=1 / 3, ferr_extra=3):
    """
    n tarefas, m = n + ferr_extra ferramentas,
    capacidade = ceil(m * fator_capacidade).
    fator_capacidade menor => instancia mais dificil (mais trocas).
    Cada tarefa usa entre 2 e 'capacidade' ferramentas; toda ferramenta
    e usada por pelo menos uma tarefa.
    """
    rng = np.random.default_rng(seed)
    m = n + ferr_extra
    capacidade = max(3, math.ceil(m * fator_capacidade))

    ferramentas = np.zeros((n, m), dtype=np.uint8)
    for i in range(n):
        k = int(rng.integers(2, capacidade + 1))
        ferramentas[i, rng.choice(m, size=k, replace=False)] = 1

    for t in range(m):
        if ferramentas[:, t].sum() == 0:
            candidatas = [
                i for i in range(n) if ferramentas[i].sum() < capacidade
            ]
            if candidatas:
                ferramentas[int(rng.choice(candidatas)), t] = 1
            else:
                # todas as tarefas cheias: troca uma ferramenta repetida
                for i in rng.permutation(n):
                    rep = [
                        u for u in np.flatnonzero(ferramentas[i])
                        if ferramentas[:, u].sum() >= 2
                    ]
                    if rep:
                        ferramentas[i, int(rng.choice(rep))] = 0
                        ferramentas[i, t] = 1
                        break

    return ferramentas, capacidade


_CACHE_MASCARAS = {"obj": None, "masks": None}


def _mascaras(ferramentas):
    if _CACHE_MASCARAS["obj"] is not ferramentas:
        masks = [
            int("".join(map(str, linha[::-1])), 2) for linha in ferramentas
        ]
        _CACHE_MASCARAS["obj"] = ferramentas
        _CACHE_MASCARAS["masks"] = masks
    return _CACHE_MASCARAS["masks"]


def custo_magazine(seq, ferramentas, capacidade):
    """
    Numero total de insercoes de ferramentas (incluindo a carga inicial)
    com a politica otima KTNS (keep tool needed soonest).
    """
    masks = _mascaras(ferramentas)
    n = len(seq)
    mag = 0
    custo = 0

    for i in range(n):
        need = masks[seq[i]]
        if need.bit_count() > capacidade:
            return float("inf")

        custo += (need & ~mag).bit_count()
        mag |= need

        excesso = mag.bit_count() - capacidade
        if excesso > 0:
            cand = mag & ~need
            manter = cand.bit_count() - excesso
            mantidas = 0
            nk = 0
            j = i + 1
            while nk < manter and j < n:
                t = cand & masks[seq[j]] & ~mantidas
                while t and nk < manter:
                    baixo = t & -t
                    mantidas |= baixo
                    t ^= baixo
                    nk += 1
                j += 1
            mag = (mag & ~cand) | mantidas

    return float(custo)


# ============================================================
# 1. Busca local e metodos classicos
# ============================================================

def busca_local(seq, ferramentas, capacidade):
    """Primeira melhora com vizinhancas de troca e de insercao."""
    atual = list(seq)
    custo_atual = custo_magazine(atual, ferramentas, capacidade)
    n = len(atual)
    melhorou = True

    while melhorou:
        melhorou = False

        for i in range(n - 1):
            for j in range(i + 1, n):
                atual[i], atual[j] = atual[j], atual[i]
                c = custo_magazine(atual, ferramentas, capacidade)
                if c < custo_atual:
                    custo_atual = c
                    melhorou = True
                else:
                    atual[i], atual[j] = atual[j], atual[i]

        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                cand = atual.copy()
                cand.insert(j, cand.pop(i))
                c = custo_magazine(cand, ferramentas, capacidade)
                if c < custo_atual:
                    atual, custo_atual = cand, c
                    melhorou = True

    return tuple(atual), custo_atual


def resolver_classico_multistart(
    ferramentas, capacidade, reinicializacoes=20, seed=0
):
    rng = np.random.default_rng(seed)
    n = ferramentas.shape[0]
    inicio = time.perf_counter()

    melhor, seq_melhor = float("inf"), None
    for _ in range(reinicializacoes):
        seq0 = tuple(int(x) for x in rng.permutation(n))
        seq, c = busca_local(seq0, ferramentas, capacidade)
        if c < melhor:
            melhor, seq_melhor = c, seq

    return {
        "custo_classico": melhor,
        "seq_classica": seq_melhor,
        "tempo_classico": time.perf_counter() - inicio,
    }


def forca_bruta(ferramentas, capacidade):
    """Enumeracao exata. A simetria de reversao do KTNS corta metade."""
    n = ferramentas.shape[0]
    inicio = time.perf_counter()
    melhor, seq_melhor = float("inf"), None

    for seq in itertools.permutations(range(n)):
        if seq[0] > seq[-1]:
            continue
        c = custo_magazine(seq, ferramentas, capacidade)
        if c < melhor:
            melhor, seq_melhor = c, seq

    return seq_melhor, melhor, time.perf_counter() - inicio


# ============================================================
# 2. Codificacao compacta: ranking de permutacoes
# ============================================================

def decodificar_ranking(rank, n):
    """Ranking 0..n!-1 -> permutacao (representacao factoradica)."""
    if not 0 <= rank < math.factorial(n):
        return None

    disponiveis = list(range(n))
    seq = []
    for i in range(n):
        fator = math.factorial(n - 1 - i)
        digito, rank = divmod(rank, fator)
        seq.append(disponiveis.pop(digito))
    return tuple(seq)


def numero_qubits_compacto(n):
    return math.ceil(math.log2(math.factorial(n)))


# ============================================================
# 3. Tabela de custos (com cache em disco)
# ============================================================

def construir_tabela_compacta(ferramentas, capacidade, cache_dir="cache"):
    n = ferramentas.shape[0]
    nq = numero_qubits_compacto(n)
    os.makedirs(cache_dir, exist_ok=True)

    h = hashlib.md5(ferramentas.tobytes()).hexdigest()[:10]
    arq = (
        f"{cache_dir}/energias_n{n}_m{ferramentas.shape[1]}"
        f"_c{capacidade}_{h}.npy"
    )

    inicio = time.perf_counter()
    if os.path.exists(arq):
        energias = np.load(arq)
    else:
        energias = np.zeros(1 << nq)
        for rank in range(math.factorial(n)):
            c = custo_magazine(
                decodificar_ranking(rank, n), ferramentas, capacidade
            )
            energias[rank] = c if np.isfinite(c) else n * ferramentas.shape[1]
        np.save(arq, energias)

    # Estados invalidos (rank >= n!) recebem o pior custo valido,
    # em vez de uma penalidade enorme que distorce E[QAOA].
    energias[math.factorial(n):] = energias[:math.factorial(n)].max()

    return energias, nq, time.perf_counter() - inicio


# ============================================================
# 4. Simulador QAOA direto em NumPy (sem circuito Qiskit)
# ============================================================

def _fwht(a):
    """Transformada de Walsh-Hadamard normalizada, O(N log N)."""
    N = a.size
    h = 1
    while h < N:
        a = a.reshape(-1, 2, h)
        s = a[:, 0, :] + a[:, 1, :]
        d = a[:, 0, :] - a[:, 1, :]
        a = np.stack((s, d), axis=1).reshape(-1)
        h *= 2
    return a / math.sqrt(N)


def _popcount(nq):
    idx = np.arange(1 << nq, dtype=np.uint32)
    pop = np.zeros(1 << nq, dtype=np.uint8)
    for b in range(nq):
        pop += ((idx >> b) & 1).astype(np.uint8)
    return pop


def estado_qaoa(params, energias, nq, pop, reps=1):
    N = 1 << nq
    psi = np.full(N, 1 / math.sqrt(N), dtype=complex)

    for camada in range(reps):
        gamma = params[2 * camada]
        beta = params[2 * camada + 1]

        # Camada de custo (diagonal)
        psi *= np.exp(-1j * gamma * energias)

        # Mixer RX(2*beta) em todos os qubits = H^n . diag . H^n,
        # com autovalor de sum(Z) igual a nq - 2k (k = popcount).
        psi = _fwht(psi)
        fases = np.exp(-1j * beta * (nq - 2 * np.arange(nq + 1)))
        psi *= fases[pop]
        psi = _fwht(psi)

    return psi


def energia_esperada_compacta(params, energias, nq, pop, reps):
    psi = estado_qaoa(params, energias, nq, pop, reps)
    probs = psi.real**2 + psi.imag**2
    return float(np.dot(probs, energias))


# ============================================================
# 5. QAOA compacto + diagnosticos + busca local
# ============================================================

def resolver_qaoa_compacto(
    ferramentas, capacidade, reps=1, reinicializacoes=3,
    maxiter=30, shots=1024, top_k=50, seed=0
):
    rng = np.random.default_rng(seed)

    energias, nq, tempo_tabela = construir_tabela_compacta(
        ferramentas, capacidade
    )
    pop = _popcount(nq)

    n = ferramentas.shape[0]
    nperm = math.factorial(n)

    validas = energias[:nperm]
    e_min = float(validas.min())            # otimo exato (tabela completa)
    e_unif = float(validas.mean())
    p_otimo_unif = float((validas == e_min).mean())

    melhor_amostras = float("inf")   # QAOA puro: so amostras
    melhor_com_top = float("inf")    # QAOA puro: amostras + top-k
    seq_qaoa = None
    melhor_hibrido, seq_hibrido = float("inf"), None
    melhor_aleatorio = float("inf")  # baseline: sorteio uniforme

    diag = {"e_qaoa": [], "p_otimo": [], "p_invalido": []}

    inicio = time.perf_counter()

    for _ in range(reinicializacoes):
        x0 = np.empty(2 * reps)
        for c in range(reps):
            x0[2 * c] = rng.uniform(0, 2 * np.pi)
            x0[2 * c + 1] = rng.uniform(0, np.pi / 2)

        res = minimize(
            energia_esperada_compacta, x0,
            args=(energias, nq, pop, reps),
            method="COBYLA",
            options={"maxiter": maxiter, "tol": 1e-3},
        )

        psi = estado_qaoa(res.x, energias, nq, pop, reps)
        probs = psi.real**2 + psi.imag**2
        probs /= probs.sum()

        # --- diagnosticos da distribuicao ---
        diag["e_qaoa"].append(float(np.dot(probs, energias)))
        diag["p_otimo"].append(float(probs[:nperm][validas == e_min].sum()))
        diag["p_invalido"].append(float(probs[nperm:].sum()))

        # --- candidatos ---
        amostras = set(map(int, rng.choice(len(probs), size=shots, p=probs)))
        k = min(top_k, len(probs))
        top = set(map(int, np.argpartition(probs, -k)[-k:]))
        candidatos = amostras | top

        for rank in candidatos:
            if rank >= nperm:
                continue
            seq = decodificar_ranking(rank, n)
            custo = float(energias[rank])   # igual a custo_magazine

            if rank in amostras and custo < melhor_amostras:
                melhor_amostras = custo
            if custo < melhor_com_top:
                melhor_com_top, seq_qaoa = custo, seq

            seq_l, custo_l = busca_local(seq, ferramentas, capacidade)
            if custo_l < melhor_hibrido:
                melhor_hibrido, seq_hibrido = custo_l, seq_l

        # --- baseline aleatorio com o mesmo numero de avaliacoes ---
        ranks_rand = rng.integers(0, nperm, size=len(candidatos))
        melhor_aleatorio = min(
            melhor_aleatorio, float(energias[ranks_rand].min())
        )

    tempo_otim = time.perf_counter() - inicio
    return {
        "qubits": nq,
        "otimo_tabela": e_min,
        "e_uniforme": e_unif,
        "p_otimo_uniforme": p_otimo_unif,
        "e_qaoa": float(np.mean(diag["e_qaoa"])),
        "p_otimo_qaoa": float(np.mean(diag["p_otimo"])),
        "p_invalido": float(np.mean(diag["p_invalido"])),
        "custo_qaoa_amostras": melhor_amostras,
        "custo_qaoa": melhor_com_top,
        "seq_qaoa": seq_qaoa,
        "custo_hibrido": melhor_hibrido,
        "seq_hibrido": seq_hibrido,
        "custo_aleatorio": melhor_aleatorio,
        "tempo_tabela": tempo_tabela,
        "tempo_otimizacao": tempo_otim,
        "tempo_total": tempo_tabela + tempo_otim,
    }


# ============================================================
# 6. Experimento comparativo
# ============================================================

def executar_comparacao(
    n,
    seed_instancia,
    seed=0,
    rodar_qaoa=True,
    reps=1,
    maxiter=20,
    reinicializacoes_qaoa=2,
    reinicializacoes_classicas=20,
    rodar_forca_bruta=True,
    fator_capacidade=1 / 3,
):
    ferramentas, capacidade = gerar_instancia(
        n, seed=seed_instancia, fator_capacidade=fator_capacidade
    )

    print(f"\n{'=' * 60}")
    print(f"INSTANCIA: {n} tarefas (semente {seed_instancia})")
    print(f"Ferramentas: {ferramentas.shape[1]} | Capacidade: {capacidade}")

    r = {
        "Tarefas": n,
        "Semente": seed_instancia,
        "Ferramentas": ferramentas.shape[1],
        "Capacidade": capacidade,
        "Qubits compactos": numero_qubits_compacto(n),
    }

    # Busca local classica
    c = resolver_classico_multistart(
        ferramentas, capacidade,
        reinicializacoes=reinicializacoes_classicas, seed=seed,
    )
    r["Custo busca local"] = c["custo_classico"]
    r["Tempo busca local (s)"] = c["tempo_classico"]

    # Otimo exato por enumeracao
    opt = None
    if rodar_forca_bruta:
        seq_opt, opt, t_bruta = forca_bruta(ferramentas, capacidade)
        r["Otimo exato"] = opt
        r["Tempo forca bruta (s)"] = t_bruta
        r["Sequencia otima"] = str(seq_opt)

    # QAOA
    if rodar_qaoa:
        q = resolver_qaoa_compacto(
            ferramentas, capacidade, reps=reps,
            reinicializacoes=reinicializacoes_qaoa,
            maxiter=maxiter, shots=512, seed=seed,
        )
        if opt is None:
            opt = q["otimo_tabela"]       # a tabela ja contem o otimo
            r["Otimo exato"] = opt

        r["Custo minimo possivel (m)"] = ferramentas.shape[1]
        r["E[uniforme]"] = q["e_uniforme"]
        r["E[QAOA]"] = q["e_qaoa"]
        r["P(otimo) uniforme"] = q["p_otimo_uniforme"]
        r["P(otimo) QAOA"] = q["p_otimo_qaoa"]
        r["Razao P(otimo) QAOA/uniforme"] = (
            q["p_otimo_qaoa"] / q["p_otimo_uniforme"]
        )
        r["P(invalido)"] = q["p_invalido"]
        r["Custo aleatorio (mesmas avaliacoes)"] = q["custo_aleatorio"]
        r["Custo QAOA (so amostras)"] = q["custo_qaoa_amostras"]
        r["Custo QAOA (amostras+top)"] = q["custo_qaoa"]
        r["Custo QAOA hibrido"] = q["custo_hibrido"]
        r["Tempo tabela (s)"] = q["tempo_tabela"]
        r["Tempo QAOA (s)"] = q["tempo_otimizacao"]
        r["Sequencia QAOA"] = str(q["seq_qaoa"])
        r["Sequencia hibrida"] = str(q["seq_hibrido"])

    if opt is not None and opt > 0:
        r["Gap busca local (%)"] = 100 * (c["custo_classico"] - opt) / opt
        if rodar_qaoa:
            for chave, nome in [
                ("Custo aleatorio (mesmas avaliacoes)", "Gap aleatorio (%)"),
                ("Custo QAOA (so amostras)", "Gap QAOA amostras (%)"),
                ("Custo QAOA (amostras+top)", "Gap QAOA amostras+top (%)"),
                ("Custo QAOA hibrido", "Gap QAOA hibrido (%)"),
            ]:
                r[nome] = 100 * (r[chave] - opt) / opt

        print(
            f"otimo={opt:.0f}  busca local={c['custo_classico']:.0f}"
            + (
                f"  aleatorio={r['Custo aleatorio (mesmas avaliacoes)']:.0f}"
                f"  QAOA(amostras)={r['Custo QAOA (so amostras)']:.0f}"
                f"  E[QAOA]={r['E[QAOA]']:.2f} vs E[unif]={r['E[uniforme]']:.2f}"
                f"  P(otimo) x{r['Razao P(otimo) QAOA/uniforme']:.2f}"
                if rodar_qaoa else ""
            )
        )

    return r


def resumir(df):
    """Media por numero de tarefas das metricas principais."""
    cols = [
        c for c in df.columns
        if c.startswith("Gap") or c in (
            "E[uniforme]", "E[QAOA]", "Razao P(otimo) QAOA/uniforme",
            "P(invalido)", "Tempo QAOA (s)", "Tempo forca bruta (s)",
            "Tempo busca local (s)",
        )
    ]
    return df.groupby("Tarefas")[cols].mean().round(3)


if __name__ == "__main__":
    # --- configuracao ---
    N_LISTA = range(5, 11)
    # Quantas instancias (sementes) por tamanho. n=9 e 10 sao caros.
    SEMENTES_POR_N = {5: 5, 6: 5, 7: 5, 8: 3, 9: 1, 10: 1}
    FATOR_CAPACIDADE = 1 / 3        # use 0.5 para o gerador mais facil
    RODAR_QAOA_ATE = 10
    REPS = 1                        # camadas do QAOA (p)
    MAXITER = 20
    REINIC_QAOA = 2
    FORCA_BRUTA_ATE = 10            # reduza (ex.: 9) para economizar tempo

    resultados = []
    for n in N_LISTA:
        for s in range(SEMENTES_POR_N.get(n, 1)):
            resultados.append(
                executar_comparacao(
                    n=n,
                    seed_instancia=1000 * n + s,
                    seed=300 + n + s,
                    rodar_qaoa=(n <= RODAR_QAOA_ATE),
                    reps=REPS,
                    maxiter=MAXITER,
                    reinicializacoes_qaoa=REINIC_QAOA,
                    reinicializacoes_classicas=20,
                    rodar_forca_bruta=(n <= FORCA_BRUTA_ATE),
                    fator_capacidade=FATOR_CAPACIDADE,
                )
            )

    df = pd.DataFrame(resultados)
    df.to_csv("comparacao_ssp_qaoa_detalhado.csv", index=False)

    resumo = resumir(df)
    resumo.to_csv("comparacao_ssp_qaoa_resumo.csv")

    print("\n=== RESUMO POR TAMANHO (media das sementes) ===")
    print(resumo.to_string())


INSTANCIA: 5 tarefas
Ferramentas: 8
Capacidade: 4

INSTANCIA: 6 tarefas
Ferramentas: 9
Capacidade: 5

INSTANCIA: 7 tarefas
Ferramentas: 10
Capacidade: 5

INSTANCIA: 8 tarefas
Ferramentas: 11
Capacidade: 6

INSTANCIA: 9 tarefas
Ferramentas: 12
Capacidade: 6

INSTANCIA: 10 tarefas
Ferramentas: 13
Capacidade: 7

=== RESULTADOS SSP/MTSP ===
 Tarefas  Ferramentas  Capacidade  Qubits compactos  Custo busca local  Tempo busca local (s)             Sequencia classica  Otimo exato  Tempo forca bruta (s)                Sequencia otima  Custo QAOA  Custo QAOA hibrido  Tempo tabela (s)  Tempo QAOA (s)                 Sequencia QAOA              Sequencia hibrida  Gap classico (%)  Gap QAOA hibrido (%)
       5            8           4                 7                8.0               0.003916                (0, 2, 1, 3, 4)          8.0               0.000222                (0, 2, 1, 3, 4)         8.0                 8.0          0.002441        0.056564                (0, 2, 1, 3, 4)            